<a href="https://colab.research.google.com/github/hadri1-bisch/SCM/blob/main/scm_survey_part_3_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

cModélisation du questionnaire:

In [ ]:
import itertools
import pandas as pd
import numpy as np


In [ ]:
# Définition des niveaux d'attributs
prix = [100, 150, 300]
hygiene = [1, 2]     # 1: seconde main, 2: standard
assortiment = [1, 2] # 1: Limité, 2: Extensif
garantie = [1, 2]    # 1: Sans retour, 2: Retours 30j

# Génération des 24 profils uniques
profils = list(itertools.product(prix, hygiene, assortiment, garantie))
df_profils = pd.DataFrame(profils, columns=['Prix', 'Hygiene', 'Assortiment', 'Garantie'])
df_24_profils = pd.DataFrame(profils, columns=['Prix', 'Hygiène', 'Assortiment', 'Garantie'])
print(df_24_profils.to_string())

    Prix  Hygiène  Assortiment  Garantie
0    100        1            1         1
1    100        1            1         2
2    100        1            2         1
3    100        1            2         2
4    100        2            1         1
5    100        2            1         2
6    100        2            2         1
7    100        2            2         2
8    150        1            1         1
9    150        1            1         2
10   150        1            2         1
11   150        1            2         2
12   150        2            1         1
13   150        2            1         2
14   150        2            2         1
15   150        2            2         2
16   300        1            1         1
17   300        1            1         2
18   300        1            2         1
19   300        1            2         2
20   300        2            1         1
21   300        2            1         2
22   300        2            2         1
23   300        

trie des pairs

In [ ]:
# Vous prenez votre tableau existant (df_24_profils)
# et vous supprimez DIRECTEMENT les lignes impossibles (2, 3, 10, 11, 18, 19) :
df_valide = df_24_profils.drop(index=[2, 3, 10, 11,16,17, 18, 19]).reset_index(drop=True)

# Afficher votre nouveau DataFrame propre
print(df_valide.to_string())

    Prix  Hygiène  Assortiment  Garantie
0    100        1            1         1
1    100        1            1         2
2    100        2            1         1
3    100        2            1         2
4    100        2            2         1
5    100        2            2         2
6    150        1            1         1
7    150        1            1         2
8    150        2            1         1
9    150        2            1         2
10   150        2            2         1
11   150        2            2         2
12   300        2            1         1
13   300        2            1         2
14   300        2            2         1
15   300        2            2         2


In [ ]:
pip install choicedesign

In [ ]:
import choicedesign as cd

mettre plus de conditions restrictives (EX: pas de prix de seconde main)

In [ ]:
# 2. Génération de la grille complète des 24 profils possibles
profils = list(itertools.product(prix, hygiene, assortiment, garantie))
df_profils = pd.DataFrame(profils, columns=['Prix', 'Hygiène', 'Assortiment', 'Garantie'])

# 3. Tirage aléatoire contrôlé de 9 paires (reproductible grâce à seed)
np.random.seed(42)
indices_a = np.random.choice(len(df_profils), size=9, replace=False)
indices_b = np.random.choice(len(df_profils), size=9, replace=False)

# 4. Construction des 16 scénarios (Option A vs Option B)
paires = []
for i in range(9):
    opt_a = df_profils.iloc[indices_a[i]].to_dict()
    opt_b = df_profils.iloc[indices_b[i]].to_dict()

    # S'assurer que l'Option A et l'Option B sont différentes
    while opt_a == opt_b:
        opt_b = df_profils.sample(1).iloc[0].to_dict()

    paires.append({
        'Scénario': f"Question {i+1}",
        'Prix_A': opt_a['Prix'], 'Hygiène_A': opt_a['Hygiène'], 'Assortiment_A': opt_a['Assortiment'], 'Garantie_A': opt_a['Garantie'],
        'Prix_B': opt_b['Prix'], 'Hygiène_B': opt_b['Hygiène'], 'Assortiment_B': opt_b['Assortiment'], 'Garantie_B': opt_b['Garantie']
    })

# 5. Création du tableau final
df_cartes = pd.DataFrame(paires)

# Affichage des cartes de choix générées
print(df_cartes.to_string())



     Scénario  Prix_A  Hygiène_A  Assortiment_A  Garantie_A  Prix_B  Hygiène_B  Assortiment_B  Garantie_B
0  Question 1     150          1              1           1     100          2              1           2
1  Question 2     300          1              1           1     300          1              2           1
2  Question 3     100          1              1           1     100          1              1           2
3  Question 4     300          1              2           1     100          2              2           2
4  Question 5     150          1              2           2     100          1              1           1
5  Question 6     150          1              1           2     150          1              2           1
6  Question 7     150          2              1           2     150          2              1           1
7  Question 8     100          1              1           2     300          1              2           2
8  Question 9     300          2              

In [ ]:
# Vérifier l'équilibre des prix dans l'Option A et l'Option B
print("--- Répartition des niveaux de Prix (Option A) ---")
print(df_cartes['Prix_A'].value_counts())

print("\n--- Répartition des niveaux de Prix (Option B) ---")
print(df_cartes['Prix_B'].value_counts())

--- Répartition des niveaux de Prix (Option A) ---
Prix_A
150    4
300    3
100    2
Name: count, dtype: int64

--- Répartition des niveaux de Prix (Option B) ---
Prix_B
100    5
300    2
150    2
Name: count, dtype: int64


In [ ]:
import pandas as pd

# 1. On crée une copie de travail pour transformer le texte en chiffres
# (nécessaire pour calculer une corrélation mathématique)
df_corr = df_cartes.copy()

# Dictionnaires de conversion textuelle -> numérique
mapping_hygiene = {'Standard': 0, 'Premium': 1}
mapping_assortiment = {'Limité': 0, 'Extensif': 1}
mapping_garantie = {'SansRetour': 0, 'Garantie30j': 1}

for col in ['Hygiène_A', 'Hygiène_B']:
    df_corr[col] = df_corr[col].map(mapping_hygiene)
for col in ['Assortiment_A', 'Assortiment_B']:
    df_corr[col] = df_corr[col].map(mapping_assortiment)
for col in ['Garantie_A', 'Garantie_B']:
    df_corr[col] = df_corr[col].map(mapping_garantie)

# 2. Calcul de la matrice de corrélation globale
matrice_correlation = df_corr.corr()

print("--- Matrice de Corrélation des Attributs ---")
print(matrice_correlation.round(2))

ValueError: could not convert string to float: 'Question 1'